# 37 · Checkpoint: operate the mock cluster
Level L5 · Part 5 · core · ~25 min · builds on 33, 34, 35, 36

Time: about 25 min. This checkpoint starts **three SIMULATED vLLM replicas** in Docker Compose (the first start builds the image). They run no model. They take time, refuse work when a queue limit is full (HTTP 503), and expose vLLM metric names. You kill one replica under load, then read `/metrics` to prove the fleet recovered.

Without Docker, every table comes from the **recorded output** (`data/recorded/37.json`), a live run of this same notebook on the course machine. The recorded output is labelled as such.

Ports: 8370 to 8372 (replicas r1 to r3, container port 8000). RAM: about 1 GB for the three containers. To run it live, set `TUT_ENABLE=docker` before you start Jupyter.

## Goal
By the end you can run a three-node Compose cluster under load, kill one replica, and show from `/metrics` and the client's counts that the fleet recovered: the survivors kept answering, refusals were counted, the restarted replica took traffic again, and the queues drained. You decide pass or fail with a gate written before the drill.

## Why it matters
- **Without a drill:** "we survive a node loss" is a sentence in a design doc. The first real loss tests the queue limit, the retries and the restart at once. Users see timeouts for minutes, and nobody can say how many requests failed or whether the fleet ever came back.
- **With a drill:** you kill a replica on purpose, under a known load, and the answer is numbers: failed calls, refusals counted by the engine, a replica back in rotation, a queue at zero. A gate turns those numbers into PASS or FAIL.
- **When you don't need it:** a throwaway prototype with no users. Any fleet with an SLO needs the drill before its first real failure.

## The idea
The checkpoint joins four lessons. **33** gives the queue limit (a full queue answers 503 at once), the retry onto another replica and the counter `vllm:admission_rejections_total`. **34** warns that in a split prefill and decode fleet a dead node loses different work; these replicas are not split. **35** gives the eval gate: fix the pass rule before the change. **36** says what a report may hold: counts and labels, never prompt text.

Think of a fire drill: sound the alarm on a normal shift, count who got out, then check the list.

```mermaid
flowchart LR
  C["Client: 120 requests, 16 in flight"] --> G["Gateway: next replica, retry on 503 or a dead node"]
  G --> R1["r1: queue limit 8"]
  G --> R2["r2: killed under load, restarted"]
  G --> R3["r3: queue limit 8"]
```

**Interview:** "A replica dies under load. How do you show the fleet recovered?" Strong answer: a gate set before the drill with numbers you can read: zero failed requests, refusals counted only on the survivors, the restarted replica serving again, and queues back to zero. Each number comes from the client's counts or from `/metrics`, never from a dashboard someone describes.

## Requirements
The gate (Build it, step 7) checks these eight numbers from the run.

1. **Healthy, no refusals:** before the kill, no 503s and no failed calls.
2. **Healthy, spread:** all three replicas serve traffic.
3. **Node loss, answered:** every request is answered (failed calls = 0) while one replica is killed under load.
4. **Refusals are counted:** the 503s the client saw equal the rise of `vllm:admission_rejections_total` on the survivors; the killed replica has no reading.
5. **Restart:** the killed replica answers `/health` again and serves traffic after the restart.
6. **Recovered:** no refusals and no failed calls after the restart.
7. **Drained:** no running or waiting requests on any replica at the end.
8. **Privacy:** the stored results hold counts only, no prompt text.

Item 3 is the one real fleets miss: the survivors have less room, so size the load for N minus one (lesson 33).

## Build it
Each step is one cell. The setup and the helpers run anywhere. Steps 2, 5 and 8 start and kill containers, so they need Docker; without it, Step 6 reads the recording.

### Step 1: the setup
The names, the replica addresses and the recorded run. Nothing here starts a container.

In [1]:
import asyncio, atexit, itertools, json, random, subprocess, time
from pathlib import Path
import httpx
from prometheus_client.parser import text_string_to_metric_families

PROJECT, COMPOSE = "tut-kv-forge-15h-37", "../infra/compose/checkpoint-37-3node.yml"
REPLICAS = {"r1": "http://127.0.0.1:8370", "r2": "http://127.0.0.1:8371", "r3": "http://127.0.0.1:8372"}
MODEL = "brightside-qwen-0.5b"                       # the model name the simulated replicas answer to
COOLDOWN_S, TRIES, IN_FLIGHT, N_REQ = 5.0, 4, 16, 120   # 16 = two survivors x queue limit 8
REC_FILE = Path("../data/recorded/37.json")                       # the recorded live run (Docker-free)
REC = json.loads(REC_FILE.read_text()) if REC_FILE.exists() else None
LIVE, RESULTS = False, {}
print("lab:", PROJECT, "| SIMULATED replicas, no model runs | recorded run:", REC["recorded"] if REC else "none yet")

lab: tut-kv-forge-15h-37 | SIMULATED replicas, no model runs | recorded run: 2026-10-23


### Step 1b: the helpers
`compose` runs a Compose command for this lab. `wait_health` polls `/health` with a timeout. `snap` reads one replica's `/metrics` page and returns the few numbers the gate needs.

In [2]:
def compose(*args):
    r = subprocess.run(["docker", "compose", "-p", PROJECT, "-f", COMPOSE, *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-800:])

def wait_health(url, limit=180):
    """Seconds until the replica's /health answers 200; raises after the limit."""
    t0 = time.perf_counter()
    for _ in range(limit):
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200: return round(time.perf_counter() - t0, 1)
        except httpx.TransportError: pass
        time.sleep(1)
    raise TimeoutError(url)

def snap(text):
    """The numbers the gate reads from one /metrics page. The page adds _total to counters."""
    s = {"running": 0, "waiting": 0, "kv": 0.0, "success": 0, "rejections": 0}
    for f in text_string_to_metric_families(text):
        for x in f.samples:
            if x.name == "vllm:num_requests_running": s["running"] = int(x.value)
            elif x.name == "vllm:num_requests_waiting": s["waiting"] = int(x.value)
            elif x.name == "vllm:kv_cache_usage_perc": s["kv"] = round(x.value, 2)
            elif x.name == "vllm:request_success_total": s["success"] = int(x.value)
            elif x.name == "vllm:admission_rejections_total" and x.labels.get("reason") == "max_num_queued_reqs":
                s["rejections"] = int(x.value)
    return s
print("helpers ready")

helpers ready


### Step 2: start three replicas
The Compose file starts r1 to r3, each with a queue limit of 8 and 4 decode slots. The cell waits until all three answer `/health`. `LIVE` becomes true only then, and the containers stop at the end of the notebook even if a cell fails.

In [3]:
compose("up", "-d", "--build")
for url in REPLICAS.values(): wait_health(url)
LIVE = True
atexit.register(lambda: LIVE and compose("down", "-v"))
print("three SIMULATED replicas answer /health")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 3: the client, a gateway with a retry rule
The workload is fixed (seed 37): 120 requests, each a 200-word policy prompt and a 20 to 60 word question. The client keeps 16 in flight. Over three replicas that is five or six each, under the limit of 8. Over two survivors it is eight each: the full limit. A retry that lands while a queue is full gets a 503, which the engine counts. The gateway sends each call to the next replica that is not cooling down. A 503 moves the call to another replica after a backoff. A connection error cools the dead replica down for five seconds.

In [4]:
rng = random.Random(37)
POLICY = [f"clause-{i}" for i in range(200)]
WORDS = [POLICY + [f"q{rng.randrange(500)}" for _ in range(rng.randint(20, 60))] for _ in range(N_REQ)]
print("workload:", len(WORDS), "requests, each", len(POLICY), "policy words + a question")

workload: 120 requests, each 200 policy words + a question


The gateway picks the next replica that is not cooling down. A dead replica cools down for five seconds.

In [5]:
class Gateway:
    def __init__(self):
        self.turn, self.cool = itertools.cycle(REPLICAS), dict.fromkeys(REPLICAS, 0.0)
    def pick(self):
        for _ in range(len(REPLICAS)):
            r = next(self.turn)
            if self.cool[r] <= time.monotonic(): return r
        return None

print("gateway ready:", list(REPLICAS), "| try", TRIES, "times")

gateway ready: ['r1', 'r2', 'r3'] | try 4 times


`send` makes one call and applies the rule: a 200 ends it, a 503 or a dead node moves the call on.

In [6]:
async def send(client, gw, words, stats):
    body = {"model": MODEL, "max_tokens": 32, "messages": [{"role": "user", "content": " ".join(words)}]}
    for attempt in range(TRIES):
        r = gw.pick()
        if r is None: break
        stats["calls"] += 1
        try:
            resp = await client.post(REPLICAS[r] + "/v1/chat/completions", json=body, timeout=30)
        except httpx.TransportError:
            gw.cool[r] = time.monotonic() + COOLDOWN_S; stats["conn_error"] += 1; continue
        if resp.status_code == 200:
            stats["ok"] += 1; stats["served"][r] += 1
            if attempt: stats["retried_ok"] += 1
            return
        stats["503"] += 1                                  # refused at the door: the queue is full
        await asyncio.sleep(0.2 * (attempt + 1))           # a longer backoff each try, not an immediate loop
    stats["failed"] += 1
print("retry rule:", TRIES, "tries, backoff 0.2 s x try number, cooldown", COOLDOWN_S, "s,", N_REQ, "requests")

retry rule: 4 tries, backoff 0.2 s x try number, cooldown 5.0 s, 120 requests


### Step 4: read every replica
`snapshot` reads `/metrics` from each replica that answers. A killed replica gives `None`: it is down, not zero. `run_phase` sends the 120 requests and, when asked, kills one replica after 40 requests have finished, the way a node failure would. It returns the client's counts, the metric snapshots before and after, and the rise of the rejection counter on each replica.

In [7]:
def snapshot():
    out = {}
    for r, url in REPLICAS.items():
        try: out[r] = snap(httpx.get(url + "/metrics", timeout=5).text)
        except httpx.TransportError: out[r] = None
    return out

`run_phase` sends the 120 requests. With `kill`, it stops that replica after 40 requests have finished.

In [8]:
async def run_phase(kill=None):
    stats = {"calls": 0, "ok": 0, "retried_ok": 0, "503": 0, "conn_error": 0, "failed": 0,
             "served": dict.fromkeys(REPLICAS, 0), "killed": None}
    before, gw, gate, done = snapshot(), Gateway(), asyncio.Semaphore(IN_FLIGHT), [0]
    async with httpx.AsyncClient() as client:
        async def one(words):
            async with gate:
                await send(client, gw, words, stats)
                done[0] += 1
                if kill and done[0] == 40 and stats["killed"] is None:
                    stats["killed"] = kill; await asyncio.to_thread(compose, "kill", kill)   # SIGKILL: a crash
        await asyncio.gather(*(one(w) for w in WORDS))
    after = snapshot()
    rise = {r: after[r]["rejections"] - before[r]["rejections"] if before[r] and after[r] else None for r in REPLICAS}
    stats.update(rejected_by=rise, rejected=sum(v for v in rise.values() if v), snap_before=before, snap_after=after)
    return stats
print("drill helpers ready")

drill helpers ready


### Step 5: the drill
Three phases, one after another. **Healthy:** all replicas up. **Node loss:** r2 is killed after 40 requests have finished. **Recovered:** r2 is started again and rejoins. Without Docker this cell does nothing, and Step 6 reads the recording.

In [9]:
if LIVE:
    RESULTS["healthy"] = await run_phase()
    RESULTS["node loss"] = await run_phase(kill="r2")
    compose("start", "r2"); wait_health(REPLICAS["r2"])
    RESULTS["recovered"] = await run_phase()
    print("drill finished:", {k: v["calls"] for k, v in RESULTS.items()}, "calls per phase")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 6: read the drill
The tables come from this live run if Docker ran, and from the recording otherwise. Every number here comes from a SIMULATED replica: no model ran. `calls` counts every try, retries included. A replica's column is the requests it answered in that phase.

In [10]:
R = RESULTS if LIVE else REC["phases"]
print("source:", "this live run (SIMULATED replicas, no model ran)" if LIVE
      else "recorded output (a live run of this notebook; SIMULATED replicas, no model ran)")
print(f"{'phase':<10} {'calls':>5} {'ok':>4} {'503s':>5} {'retried ok':>10} {'failed':>6} {'conn err':>8}  {'r1':>4} {'r2':>4} {'r3':>4}  {'rejected':>8}")
for k, s in R.items():
    print(f"{k:<10} {s['calls']:>5} {s['ok']:>4} {s['503']:>5} {s['retried_ok']:>10} {s['failed']:>6} {s['conn_error']:>8}  "
          f"{s['served']['r1']:>4} {s['served']['r2']:>4} {s['served']['r3']:>4}  {s['rejected']:>8}")

def cell(snapshot_, r, key):
    s = snapshot_.get(r)
    return "down" if s is None else str(s[key])

print("\nWhat the replicas say (/metrics), at the end of each phase:")
print(f"{'replica':<8} {'healthy: success':>17} {'node loss: success':>19} {'recovered: success':>19} {'running/waiting now':>20}")
END = R["recovered"]["snap_after"]
for r in REPLICAS:
    print(f"{r:<8} {cell(R['healthy']['snap_after'], r, 'success'):>17} {cell(R['node loss']['snap_after'], r, 'success'):>19} "
          f"{cell(R['recovered']['snap_after'], r, 'success'):>19} {cell(END, r, 'running') + '/' + cell(END, r, 'waiting'):>20}")
print("\nrefusals per replica in the node-loss phase (the rise of vllm:admission_rejections_total):", R["node loss"]["rejected_by"])
print("note: r2 restarted as a new process, so its counter starts again from 0 after the restart.")

source: recorded output (a live run of this notebook; SIMULATED replicas, no model ran)
phase      calls   ok  503s retried ok failed conn err    r1   r2   r3  rejected
healthy      120  120     0          0      0        0    40   40   40         0
node loss    128  120     3          8      0        5    52   16   52         3
recovered    120  120     0          0      0        0    40   40   40         0

What the replicas say (/metrics), at the end of each phase:
replica   healthy: success  node loss: success  recovered: success  running/waiting now
r1                      40                  92                 132                  0/0
r2                      40                down                  40                  0/0
r3                      40                  92                 132                  0/0

refusals per replica in the node-loss phase (the rise of vllm:admission_rejections_total): {'r1': 2, 'r2': None, 'r3': 1}
note: r2 restarted as a new process, so its counter 

### Step 7: write the gate
Write the gate for the eight items in Requirements. Build a list of `(text, passed)` pairs from `R`, print each one as PASS or FAIL, and print how many passed. Use the phase names in `R`: `"healthy"`, `"node loss"`, `"recovered"`. The killed replica's name is `R["node loss"]["killed"]`. The solution is in the Reference solution section, after the stop step, so try it first.

In [11]:
# your code here
checks = []

### Step 8: stop the cluster
Save the live run as a run-time file (not committed) and stop the containers. Without Docker this cell does nothing.

In [12]:
if LIVE:
    Path("../data/37").mkdir(parents=True, exist_ok=True)
    Path("../data/37/results.json").write_text(json.dumps({"phases": RESULTS}, indent=1))
    print("saved the live run to data/37/results.json (run-time file, not committed)")
    compose("down", "-v"); LIVE = False
    print("cluster stopped")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## Reference solution
Try the gate yourself first (Step 7). When you are ready, expand the hidden cell below. It applies the eight Requirements to `R` and prints the verdict. It works on the live run and on the recording.

In [13]:
H, NL, RC = R["healthy"], R["node loss"], R["recovered"]
K = NL["killed"]
checks = [
    ("1 healthy, no refusals: no 503s, no failed calls", H["503"] == 0 and H["failed"] == 0),
    ("2 healthy, spread: all three replicas served traffic", all(v > 0 for v in H["served"].values())),
    ("3 node loss, answered: every request answered (failed = 0)", NL["failed"] == 0),
    ("4 refusals counted: the 503s the client saw = the survivors' counter rise, none on the killed replica",
     NL["rejected_by"][K] is None and NL["rejected"] == NL["503"]),
    ("5 restart: the killed replica answers /health and serves traffic", RC["snap_before"][K] is not None and RC["served"][K] > 0),
    ("6 recovered: no refusals, no failed calls", RC["503"] == 0 and RC["failed"] == 0),
    ("7 drained: no running or waiting requests on any replica at the end",
     all(s is not None and s["running"] == 0 and s["waiting"] == 0 for s in RC["snap_after"].values())),
    ("8 privacy: the stored results hold counts only, no prompt text", "clause-" not in json.dumps(R)),
]
for text, ok in checks: print("PASS" if ok else "FAIL", " ", text)
print(f"gate: {sum(ok for _, ok in checks)} of {len(checks)} passed (SIMULATED fleet, recorded or live as shown in Step 6)")

PASS   1 healthy, no refusals: no 503s, no failed calls
PASS   2 healthy, spread: all three replicas served traffic
PASS   3 node loss, answered: every request answered (failed = 0)
PASS   4 refusals counted: the 503s the client saw = the survivors' counter rise, none on the killed replica
PASS   5 restart: the killed replica answers /health and serves traffic
PASS   6 recovered: no refusals, no failed calls
PASS   7 drained: no running or waiting requests on any replica at the end
PASS   8 privacy: the stored results hold counts only, no prompt text
gate: 8 of 8 passed (SIMULATED fleet, recorded or live as shown in Step 6)


## Key terms
- **Node loss** — a replica (here, a container) stops answering in the middle of traffic; its share moves to the survivors.
- **Admission rejection (503)** — the engine refuses a request at the door because its queue is full; the counter `vllm:admission_rejections_total` counts each refusal by `reason`.
- **Drained queue** — after a drill, the queue is what you check: `vllm:num_requests_running` and `vllm:num_requests_waiting` must return to 0.
- **Gate** — a pass rule written before the change or drill, with numbers that decide it (lesson 35 applies the same idea to rollouts).
- **SIMULATED replica** — a mock engine (lesson 33) with vLLM metric names and no model; its numbers show the fleet's behaviour, not a model's speed.